In [1]:
import pandas as pd 
import numpy as np 
import os 
import time
import logging 
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error

from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor

from category_encoders import TargetEncoder

from tqdm.auto import tqdm
from itertools import combinations
import warnings
warnings.simplefilter('ignore')

In [2]:
train = pd.read_csv("/kaggle/input/playground-series-s5e5/train.csv")
test = pd.read_csv("/kaggle/input/playground-series-s5e5/test.csv")
submission = pd.read_csv("/kaggle/input/playground-series-s5e5/sample_submission.csv")

## EDA and Preprocessing

In [3]:
numerical_features = ["Age","Height","Weight","Duration","Heart_Rate","Body_Temp"]

def add_feature_cross_terms(df, numerical_features):
    df_new = df.copy()
    #df_new['BMI'] = df_new['Weight'] / ((df_new['Height'] / 100) ** 2)
    
    for i in range(len(numerical_features)):
        for j in range(i + 1, len(numerical_features)):  
            feature1 = numerical_features[i]
            feature2 = numerical_features[j]
            cross_term_name = f"{feature1}_x_{feature2}"
            df_new[cross_term_name] = df_new[feature1] * df_new[feature2]
            #cross_term_name = f"{feature1}_add_{feature2}"
            #df_new[cross_term_name] = df_new[feature1] + df_new[feature2]
            #cross_term_name = f"{feature1}_divided_{feature2}"
            #df_new[cross_term_name] = df_new[feature1] / df_new[feature2]

    return df_new

train = add_feature_cross_terms(train, numerical_features)
test = add_feature_cross_terms(test, numerical_features)


In [4]:
num_features = train.select_dtypes(include='number')

In [5]:
le = LabelEncoder()
train['Sex'] = le.fit_transform(train['Sex'])
test['Sex'] = le.transform(test['Sex'])

train["Sex"] = train["Sex"].astype("category")
test["Sex"] = test["Sex"].astype("category")

X = train.drop(columns=["id", "Calories"])
y = np.log1p(train["Calories"])
X_test = test.drop(columns=["id"])

In [6]:
train.describe()

,id,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Calories,Age_x_Height,Age_x_Weight,...,Height_x_Weight,Height_x_Duration,Height_x_Heart_Rate,Height_x_Body_Temp,Weight_x_Duration,Weight_x_Heart_Rate,Weight_x_Body_Temp,Duration_x_Heart_Rate,Duration_x_Body_Temp,Heart_Rate_x_Body_Temp
count,750000.000000,750000.000000,750000.000000,750000.000000,750000.000000,750000.000000,750000.000000,750000.000000,750000.000000,750000.000000,...,750000.000000,750000.000000,750000.000000,750000.000000,750000.000000,750000.000000,750000.000000,750000.000000,750000.000000,750000.000000
mean,374999.500000,41.420404,174.697685,75.145668,15.421015,95.483995,40.036253,88.282781,7238.379235,3128.200032,...,13299.557672,2690.808300,16679.229017,6993.894303,1156.387451,7174.893501,3008.292357,1541.562606,623.283247,3828.687447
std,216506.495284,15.175049,12.824496,13.982704,8.354095,9.449845,0.779875,62.395349,2712.869502,1334.431304,...,3407.211385,1473.626587,2047.188593,526.939776,672.877571,1517.486807,561.697333,932.453480,343.646487,437.967454
min,0.000000,20.000000,126.000000,36.000000,1.000000,67.000000,37.100000,1.000000,2700.000000,860.000000,...,5289.000000,135.000000,9983.000000,5027.400000,45.000000,3000.000000,1450.800000,67.000000,37.100000,2485.700000
25%,187499.750000,28.000000,164.000000,63.000000,8.000000,88.000000,39.600000,34.000000,4914.000000,2046.000000,...,10354.000000,1440.000000,15219.000000,6568.900000,600.000000,5980.000000,2526.300000,728.000000,317.600000,3497.400000
50%,374999.500000,40.000000,174.000000,74.000000,15.000000,95.000000,40.300000,77.000000,6920.000000,2912.000000,...,12900.000000,2669.000000,16587.000000,6987.200000,1105.000000,7029.000000,2960.000000,1455.000000,606.000000,3838.000000
75%,562499.250000,52.000000,185.000000,87.000000,23.000000,103.000000,40.700000,136.000000,9168.000000,3978.000000,...,16016.000000,3933.000000,18050.000000,7402.900000,1633.000000,8272.000000,3468.000000,2323.000000,931.500000,4171.500000
max,749999.000000,79.000000,222.000000,132.000000,30.000000,128.000000,41.500000,314.000000,16748.000000,9401.000000,...,28776.000000,6540.000000,26199.000000,9168.600000,3780.000000,15129.000000,5412.000000,3840.000000,1245.000000,5286.400000


In [7]:
FOLDS = 50
FEATURES = X.columns.tolist()

# KFold setup
kf = KFold(n_splits=FOLDS, shuffle=True, random_state=42)

# Arrays to store predictions
oof = np.zeros(len(train))
pred = np.zeros(len(test))

# Start CV loop
for i, (train_idx, valid_idx) in enumerate(kf.split(X, y)):
    print(f"\n{'#'*10} Fold {i+1} {'#'*10}")
    
    x_train = X.iloc[train_idx].copy()
    y_train = y.iloc[train_idx]
    x_valid = X.iloc[valid_idx].copy()
    y_valid = y.iloc[valid_idx]
    x_test = X_test.copy()

    # No categorical target encoding in this dataset, but you can add if needed
    
    start = time.time()

    # Train model
    model = XGBRegressor(
        device="cuda" if XGBRegressor().get_params().get("device") == "cuda" else "cpu",
        max_depth=10,
        #min_child_weight=2,
        colsample_bytree=0.75,
        subsample=0.9,
        n_estimators=2000,
        learning_rate=0.02,
        gamma=0.01, 
        max_delta_step=2,
        early_stopping_rounds=100,
        eval_metric="rmse",
        enable_categorical=True
    )

    model.fit(
        x_train, y_train,
        eval_set=[(x_valid, y_valid)],
        verbose=100
    )

    # Predict OOF and test
    oof[valid_idx] = model.predict(x_valid)
    pred += model.predict(x_test)

    rmse = np.sqrt(mean_squared_error(y_valid, oof[valid_idx]))
    print(f"Fold {i+1} RMSE: {rmse:.4f}")
    print(f"Feature engineering & training time: {time.time() - start:.1f} sec")

# Average test predictions
pred /= FOLDS

# Final RMSE
full_rmse = np.sqrt(mean_squared_error(y, oof))
print(f"\nFinal CV RMSE: {full_rmse:.4f}")


########## Fold 1 ##########
[0]	validation_0-rmse:0.95092
[100]	validation_0-rmse:0.14288
[200]	validation_0-rmse:0.06494
[300]	validation_0-rmse:0.06259
[400]	validation_0-rmse:0.06250
[471]	validation_0-rmse:0.06255
Fold 1 RMSE: 0.0625
Feature engineering & training time: 35.9 sec

########## Fold 2 ##########
[0]	validation_0-rmse:0.94322
[100]	validation_0-rmse:0.14194
[200]	validation_0-rmse:0.06349
[300]	validation_0-rmse:0.06096
[400]	validation_0-rmse:0.06091
[482]	validation_0-rmse:0.06091
Fold 2 RMSE: 0.0609
Feature engineering & training time: 36.7 sec

########## Fold 3 ##########
[0]	validation_0-rmse:0.95575
[100]	validation_0-rmse:0.14379
[200]	validation_0-rmse:0.06205
[300]	validation_0-rmse:0.05895
[400]	validation_0-rmse:0.05883
[500]	validation_0-rmse:0.05880
[600]	validation_0-rmse:0.05880
[612]	validation_0-rmse:0.05880
Fold 3 RMSE: 0.0588
Feature engineering & training time: 41.0 sec

########## Fold 4 ##########
[0]	validation_0-rmse:0.93257
[100]	validation_0

In [8]:
y_preds = np.expm1(pred)
print('predict mean :',y_preds.mean())
print('predict median :',np.median(y_preds))

y_preds = np.clip(y_preds,1,314)
print('predict mean after clip:',y_preds.mean())
print('predict median after clip:',np.median(y_preds))

submission["Calories"] = y_preds
submission.to_csv("submission.csv", index=False)
submission.head()

predict mean : 88.14926996350052
predict median : 76.39205115564113
predict mean after clip: 88.14926996350052
predict median after clip: 76.39205115564113


,id,Calories
0,750000,27.476409
1,750001,107.811517
2,750002,87.782623
3,750003,125.934348
4,750004,75.873318
